In [1]:
import psycopg2
import pandas as pd
import sys
import os
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import pytz
import pandas_ta as ta
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'browser'

controllers_path = os.path.abspath("../historic_updater")
sys.path.append(controllers_path)

from controllers import *

credentials = CredentialsManager()
pg_credentials = PgCredentialsManager(credentials)
cnx=PgConnecionManager(pg_credentials)

# Consulta SQL
query = "SELECT * FROM public.btcusdtfutures_live"

# Cargar datos en un DataFrame
df = pd.read_sql(query, cnx.connection)

# Cerrar conexión
cnx.connection.close()

df['open_time'] = pd.to_datetime(df['open_time']).dt.tz_localize(pytz.UTC)
df['year']= df.open_time.dt.year
df['month']= df.open_time.dt.month
df['year-month'] = df.open_time.dt.strftime('%Y%m').astype(int)
df['hour'] = df.open_time.dt.hour
df.sort_values(by='open_time',ascending=True,inplace=True)
df = df[df['year']>2019]

Credentials set
Connection sucessful


/tmp/ipykernel_17134/2429048444.py:30: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



In [2]:
# Strategy
def calculate_SL_TP(data, sl_thressh = 0.1,rate = 2/1):
    df = data.copy() 


    df['stop_loss'] = np.where(
        df['signal']== 'buy', df['EMA_Long'] * (1-sl_thressh),

        np.where(
        df['signal']== 'sell', df['EMA_Long'] * (1+sl_thressh),
        None)
    )

    df['take_profit'] = np.where(
        df['signal'] == 'buy', df['close'] + (df['close']-df['stop_loss']) * rate,

        np.where(
        df['signal'] == 'sell', df['close'] - (df['stop_loss']-df['close']) * rate,
        None)
    )

    return df


def unbalanced_EMADX_strategy(data, ema_short, ema_long, ema_slow, adx_thresh, rsi_up,rsi_down):
    df = data.copy()  


    df['EMA_Short'] = ta.ema(df['close'], length=ema_short)
    df['EMA_Long'] = ta.ema(df['close'], length=ema_long)
    df['EMA_Slow'] = ta.ema(df['close'], length=ema_slow)
    df['ADX'] = ta.adx(high=df['high'], low=df['low'], close=df['close'])['ADX_14']
    df['RSI'] = ta.rsi(df['close'], length=14)


    df['signal'] = np.where(
        (df['EMA_Short'] > df['EMA_Long']) & 
        (df['EMA_Long'] > df['EMA_Slow']) &
        (df['ADX'] > adx_thresh) &
        (df['RSI']<rsi_up[0])&
        (df['RSI']>rsi_up[1])&
        (df['close']>df['EMA_Short'])
        ,
        'buy',
        np.where(
            (df['EMA_Short'] < df['EMA_Long']) & 
            (df['EMA_Long'] < df['EMA_Slow']) &
            (df['ADX'] > adx_thresh) &
            (df['RSI']<rsi_down[0])&
            (df['RSI']>rsi_down[1])&
            (df['close']<df['EMA_Short'])
            ,
            'sell',
            ''
        )
    )

    return calculate_SL_TP(df)


def EMADX_strategy(data,fast_EMA=9,slow_EMA=21, long_EMA = 200 ,ADX_thresh=25,ADX_period=14):

    """
    This function receives a DataFrame with historical data (containing the columns 'High', 'Low', and 'Close'), calculates two EMAs and the ADX, and determines the trend for each candle.

    The rule is:

    - If the fast EMA > slow EMA and ADX > ADX_thresh ⇒ Uptrend.
    - If the fast EMA < slow EMA and ADX > ADX_thresh ⇒ Downtrend.
    - Otherwise, the market is considered to be in a sideways range.
    """
    df = data.copy()
    #EMA CALC default is 9 for fast and 21 for slow
    df['EMA_fast'] = ta.ema(df['close'], length=fast_EMA)
    df['EMA_slow'] = ta.ema(df['close'], length=slow_EMA)
    df['EMA_long'] = ta.ema(df['close'], length=long_EMA)
    
    # ADX
    adx_df = ta.adx(high=df['high'], low=df['low'], close=df['close'])
    colname = 'ADX_'+str(ADX_period)
    df['ADX'] = adx_df[colname]
    
    # Función para determinar la tendencia según las condiciones definidas.
    def trend_signal(row):
        if pd.isna(row['EMA_fast']) or pd.isna(row['EMA_slow']) or pd.isna(row['ADX']):
            return 'undefined'
        if row['EMA_fast'] > row['EMA_slow'] and row['ADX'] > ADX_thresh and row['EMA_fast']> row['EMA_long'] and row['EMA_slow'] > row['EMA_long']:
            return 'uptrend'
        elif row['EMA_fast'] < row['EMA_slow'] and row['ADX'] > ADX_thresh and row['EMA_fast']<row['EMA_long'] and row['EMA_slow']<row['EMA_long']:
            return 'downtrend'
        else:
            return 'sideways'
    
    df['EMADX_trend'] = df.apply(trend_signal, axis=1)
    
    return df

#df = EMADX_strategy(df,fast_EMA=9,slow_EMA=21, long_EMA = 50 ,ADX_thresh=25,ADX_period=14)
data = unbalanced_EMADX_strategy(df, ema_short=9, ema_long=21, ema_slow=55, adx_thresh=25, rsi_down=(50,30),rsi_up=(70,50))


In [9]:
filtered_df.iloc[12]

open_time                  2025-01-01 14:05:00+00:00
open                                         94346.2
high                                         94430.0
low                                          94139.1
close                                        94240.1
volume                                       1253.78
close_time                2025-01-01 14:09:59.999000
quote_asset_volume                    118250378.6305
number_of_trades                               20258
taker_buy_base_volume                        534.202
taker_buy_quote_volume                 50390766.9035
ignore                                           0.0
year                                            2025
month                                              1
year-month                                    202501
hour                                              14
EMA_Short                               93985.686192
EMA_Long                                93826.811421
EMA_Slow                                93659.

In [10]:
funds

NameError: name 'funds' is not defined

In [11]:
import sc2

def check_strategy(data):
    i = 0
    funds = []
    for index, row in data.iterrows():
        if i == 0: 
            strategy = sc2.StrategyManager(initial_signal=row['signal'],take_profit=row['take_profit'], stop_loss=row['stop_loss'])
        strategy.update_strategy(signal = row['signal'],price = row['close'] ,take_profit=row['take_profit'], stop_loss=row['stop_loss'])
        funds += [strategy.current_funds]
        
        i+=1
        print(i)
        print(funds)
    strategy.get_performance_report()
    return funds

def strategy_sumary(funds,time):
    Y = funds
    X = list(time)

    # Crear el line plot
    plt.plot(X, Y, label="Funds vs Time", color='b')  # Puedes cambiar el color si lo deseas

    # Etiquetas y título
    plt.xlabel('Time')  # Etiqueta del eje X
    plt.ylabel('Funds')  # Etiqueta del eje Y
    plt.title("Funds vs Time")  # Título del gráfico

    # Mostrar la leyenda
    plt.legend()

    # Mostrar la figura
    plt.show()

end_str = "2025-01-01 00:00:00"
start_str = "2025-03-01 00:00:00"


colombia_tz = pytz.timezone('America/Bogota')
start_local = colombia_tz.localize(pd.to_datetime(start_str))
end_local = colombia_tz.localize(pd.to_datetime(end_str))
start_utc = start_local.astimezone(pytz.UTC)
end_utc = end_local.astimezone(pytz.UTC)

# Filtrar el DataFrame
filtered_df = data[(data['open_time'] >= end_utc) & (data['open_time'] <= start_utc) & (data['signal']!= '')]

funds = check_strategy(filtered_df)
time = filtered_df.open_time

strategy_sumary(funds,time)

1
[100.0]
2
[100.0, 100.0]
3
[100.0, 100.0, 100.0]
4
[100.0, 100.0, 100.0, 100.0]
5
[100.0, 100.0, 100.0, 100.0, 100.0]
6
[100.0, 100.0, 100.0, 100.0, 100.0, 100.0]
7
[100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0]
8
[100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0]
9
[100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0]
10
[100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0]
11
[100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0]
12
[100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, -95849.344359518]


ValueError: Investment size must be positive